# 🍲 BeautifulSoup + NLP + BM25 Chatbot
### From Web Scraping → Text Preprocessing → Search Engine → Chatbot

---

## 📚 What You Will Build Today

```
🌐 Wikipedia Article
        ↓
🍲 BeautifulSoup  →  Scrape & extract clean text
        ↓
🧹 NLP Pipeline   →  Lowercase, clean, tokenize, lemmatize
        ↓
✂️  Chunking       →  Split into searchable pieces
        ↓
🔍 BM25 Index     →  Build a keyword search engine
        ↓
🤖 Chatbot        →  Ask questions, get answers from the article
```

---

## 🗂️ Notebook Structure

| Part | Topic |
|---|---|
| **Part 1** | BeautifulSoup — Basics & Important Functions |
| **Part 2** | Scraping a Wikipedia Article |
| **Part 3** | NLP Preprocessing Pipeline |
| **Part 4** | Chunking the Document |
| **Part 5** | BM25 Search Engine |
| **Part 6** | The Chatbot — Putting It All Together |

---
## ⚙️ Setup — Install Everything First

In [1]:
# Run this cell once before anything else
import subprocess

packages = ['beautifulsoup4', 'requests', 'nltk', 'rank_bm25', 'lxml']
for pkg in packages:
    subprocess.run(['pip', 'install', pkg, '-q'], capture_output=True)
    print(f'✅ {pkg} ready')

import nltk
for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'averaged_perceptron_tagger']:
    nltk.download(resource, quiet=True)

print('\n🎉 All setup complete! You are ready to go.')

✅ beautifulsoup4 ready
✅ requests ready
✅ nltk ready
✅ rank_bm25 ready
✅ lxml ready

🎉 All setup complete! You are ready to go.


---
# 🍲 PART 1 — BeautifulSoup Basics

## What is BeautifulSoup?

**BeautifulSoup** is a Python library that helps you **parse and navigate HTML/XML** documents. Think of a webpage as a messy tree of tags — BeautifulSoup lets you climb that tree and pull out exactly what you need.

```
<html>
  <body>
    <h1>Title</h1>          ← you can grab this
    <p class="intro">...</p> ← or this
    <a href="...">link</a>   ← or all links
  </body>
</html>
```

### Two things you always need:
1. `requests` — to **download** the webpage
2. `BeautifulSoup` — to **read and navigate** the HTML

In [2]:
from bs4 import BeautifulSoup

# Let's create a small fake HTML page to practice on
sample_html = """
<html>
  <head><title>My NLP Page</title></head>
  <body>
    <h1 id="main-title">Introduction to NLP</h1>
    <h2>What is NLP?</h2>
    <p class="intro">NLP stands for Natural Language Processing.</p>
    <p class="content">It helps computers understand human language.</p>
    <p class="content">Applications include chatbots, search engines, and more.</p>
    <ul>
      <li>Tokenization</li>
      <li>Stemming</li>
      <li>Lemmatization</li>
    </ul>
    <a href="https://nltk.org">NLTK Website</a>
    <a href="https://spacy.io">spaCy Website</a>
  </body>
</html>
"""

# Parse it — 'html.parser' is Python's built-in HTML parser
soup = BeautifulSoup(sample_html, 'html.parser')

print("✅ HTML parsed successfully!")
print("Type of soup object:", type(soup))

✅ HTML parsed successfully!
Type of soup object: <class 'bs4.BeautifulSoup'>


---
## 🔧 Important BeautifulSoup Functions

### Function 1: `find()` — Get the FIRST matching tag

In [3]:
# find() returns the FIRST match only

# Find by tag name
h1 = soup.find('h1')
print("First h1 tag:     ", h1)
print("h1 text only:     ", h1.text)
print("h1 text stripped: ", h1.get_text(strip=True))

print()

# Find by tag + attribute
intro_p = soup.find('p', class_='intro')
print("p with class=intro:", intro_p.text)

# Find by id
main = soup.find(id='main-title')
print("Element with id=main-title:", main.text)

# Find title tag
print("Page title:", soup.find('title').text)

First h1 tag:      <h1 id="main-title">Introduction to NLP</h1>
h1 text only:      Introduction to NLP
h1 text stripped:  Introduction to NLP

p with class=intro: NLP stands for Natural Language Processing.
Element with id=main-title: Introduction to NLP
Page title: My NLP Page


### Function 2: `find_all()` — Get ALL matching tags

In [4]:
# find_all() returns a LIST of all matches

# Find all paragraphs
all_paragraphs = soup.find_all('p')
print(f"Found {len(all_paragraphs)} paragraphs:")
for i, p in enumerate(all_paragraphs, 1):
    print(f"  {i}: {p.text}")

print()

# Find all links
all_links = soup.find_all('a')
print(f"Found {len(all_links)} links:")
for link in all_links:
    print(f"  Text: {link.text}  |  URL: {link.get('href')}")

print()

# Find all list items
items = soup.find_all('li')
print("List items:", [item.text for item in items])

# Find all headings (h1 and h2)
headings = soup.find_all(['h1', 'h2'])
print("All headings:", [h.text for h in headings])

Found 3 paragraphs:
  1: NLP stands for Natural Language Processing.
  2: It helps computers understand human language.
  3: Applications include chatbots, search engines, and more.

Found 2 links:
  Text: NLTK Website  |  URL: https://nltk.org
  Text: spaCy Website  |  URL: https://spacy.io

List items: ['Tokenization', 'Stemming', 'Lemmatization']
All headings: ['Introduction to NLP', 'What is NLP?']


### Function 3: `select()` — Use CSS Selectors (most powerful)

In [5]:
# select() uses CSS selector syntax — very powerful

# Select by class (use dot .)
content_paras = soup.select('p.content')
print("p with class=content:")
for p in content_paras:
    print(" ", p.text)

print()

# Select by id (use #)
main_title = soup.select('#main-title')
print("Element with id=main-title:", main_title[0].text)

# Select nested elements (ul > li means li directly inside ul)
list_items = soup.select('ul li')
print("\nItems inside ul:", [li.text for li in list_items])

# CSS Selector Quick Reference:
# soup.select('p')          → all <p> tags
# soup.select('.classname') → all tags with that class
# soup.select('#id')        → tag with that id
# soup.select('div p')      → all <p> inside <div>
# soup.select('a[href]')    → all <a> tags that have href attribute

p with class=content:
  It helps computers understand human language.
  Applications include chatbots, search engines, and more.

Element with id=main-title: Introduction to NLP

Items inside ul: ['Tokenization', 'Stemming', 'Lemmatization']


### Function 4: `.text`, `.get_text()`, `.get()` — Extract Content

In [6]:
tag = soup.find('p', class_='intro')

# .text — get text content (includes child tag text too)
print(".text:              ", tag.text)

# .get_text() — same but with options
print(".get_text():        ", tag.get_text())
print(".get_text(strip=True):", tag.get_text(strip=True))  # removes extra whitespace

# .string — only if the tag has exactly one string child
print(".string:            ", tag.string)

print()

# .get() — get attribute value safely
link = soup.find('a')
print("Link href via .get():  ", link.get('href'))       # safe — returns None if missing
print("Link href via ['href']:", link['href'])            # direct — raises error if missing
print("Missing attr via .get():", link.get('id', 'N/A')) # default value if not found

print()

# .attrs — get ALL attributes as a dict
h1 = soup.find('h1')
print("All attrs of h1:", h1.attrs)

.text:               NLP stands for Natural Language Processing.
.get_text():         NLP stands for Natural Language Processing.
.get_text(strip=True): NLP stands for Natural Language Processing.
.string:             NLP stands for Natural Language Processing.

Link href via .get():   https://nltk.org
Link href via ['href']: https://nltk.org
Missing attr via .get(): N/A

All attrs of h1: {'id': 'main-title'}


### Function 5: Navigating the Tree — Parent, Children, Siblings

In [7]:
# Navigate relationships between tags

p = soup.find('p', class_='intro')

# Parent — go UP the tree
print("Parent tag:", p.parent.name)

# Next sibling — go to the NEXT tag at same level
next_tag = p.find_next_sibling('p')
print("Next p sibling:", next_tag.text if next_tag else 'None')

# All siblings of same type
all_siblings = p.find_next_siblings('p')
print("All next p siblings:")
for s in all_siblings:
    print(" ", s.text)

# Children of body
body = soup.find('body')
print("\nDirect children of body:")
for child in body.children:
    if child.name:  # skip whitespace text nodes
        print(f"  <{child.name}>: {child.get_text(strip=True)[:50]}")

Parent tag: body
Next p sibling: It helps computers understand human language.
All next p siblings:
  It helps computers understand human language.
  Applications include chatbots, search engines, and more.

Direct children of body:
  <h1>: Introduction to NLP
  <h2>: What is NLP?
  <p>: NLP stands for Natural Language Processing.
  <p>: It helps computers understand human language.
  <p>: Applications include chatbots, search engines, and
  <ul>: TokenizationStemmingLemmatization
  <a>: NLTK Website
  <a>: spaCy Website


### Function 6: `get_text()` on whole page — Extract all visible text

In [8]:
# Get ALL text from the page at once
full_text = soup.get_text()
print("All text (raw):")
print(full_text[:300])

print()

# Better: with separator and strip
clean_text = soup.get_text(separator=' ', strip=True)
print("All text (cleaned):")
print(clean_text[:300])

# This is what we'll use to extract Wikipedia article text!

All text (raw):


My NLP Page

Introduction to NLP
What is NLP?
NLP stands for Natural Language Processing.
It helps computers understand human language.
Applications include chatbots, search engines, and more.

Tokenization
Stemming
Lemmatization

NLTK Website
spaCy Website




All text (cleaned):
My NLP Page Introduction to NLP What is NLP? NLP stands for Natural Language Processing. It helps computers understand human language. Applications include chatbots, search engines, and more. Tokenization Stemming Lemmatization NLTK Website spaCy Website


### Function 7: Fetching a Real Webpage with `requests`

In [9]:
import requests

# requests.get() downloads the webpage
url = "https://httpbin.org/html"  # a safe test page
response = requests.get(url)

print("Status code:", response.status_code)
# 200 = OK, 404 = Not Found, 403 = Forbidden, 500 = Server Error

print("Content type:", response.headers.get('content-type'))
print("First 200 chars of HTML:", response.text[:200])

# Now parse it
page_soup = BeautifulSoup(response.text, 'html.parser')
print("\nPage title:", page_soup.find('title'))
print("\nAll text:", page_soup.get_text(strip=True)[:300])

Status code: 200
Content type: text/html; charset=utf-8
First 200 chars of HTML: <!DOCTYPE html>
<html>
  <head>
  </head>
  <body>
      <h1>Herman Melville - Moby-Dick</h1>

      <div>
        <p>
          Availing himself of the mild, summer-cool weather that now reigned in t

Page title: None

All text: Herman Melville - Moby-DickAvailing himself of the mild, summer-cool weather that now reigned in these latitudes, and in preparation for the peculiarly active pursuits shortly to be anticipated, Perth, the begrimed, blistered old blacksmith, had not removed his portable forge to the hold again, afte


### 📌 BeautifulSoup Quick Reference Cheat Sheet

```python
soup = BeautifulSoup(html, 'html.parser')   # Parse HTML

# Finding elements
soup.find('tag')                  # First match
soup.find('tag', class_='x')      # First match with class
soup.find(id='x')                 # By id
soup.find_all('tag')              # All matches → list
soup.find_all(['h1','h2'])        # Multiple tag types
soup.select('.classname')         # By CSS class
soup.select('#id')                # By CSS id
soup.select('div p')              # Nested: p inside div

# Extracting content
tag.text                          # Raw text
tag.get_text(strip=True)          # Clean text
tag.get('href')                   # Attribute value (safe)
tag['href']                       # Attribute value (direct)
tag.attrs                         # All attributes as dict

# Navigation
tag.parent                        # Parent element
tag.find_next_sibling('p')        # Next sibling of type
tag.children                      # Direct children

# Whole page text
soup.get_text(separator=' ', strip=True)
```

---
# 🌐 PART 2 — Scraping a Wikipedia Article

Now we use BeautifulSoup for real. We will scrape a Wikipedia article and extract **clean text** from it.

## How Wikipedia HTML is structured:

```
<div id="mw-content-text">       ← main article container
  <div class="mw-parser-output">  ← actual content
    <p>First paragraph...</p>     ← article paragraphs
    <h2>Section heading</h2>       ← section titles
    <p>More content...</p>
  </div>
</div>
```

> 📝 You can change `WIKI_URL` below to scrape any Wikipedia article you like!

In [10]:
import requests
from bs4 import BeautifulSoup

# ✏️ Change this to any Wikipedia article you want!
WIKI_URL = "https://en.wikipedia.org/wiki/Natural_language_processing"

# Step 1: Download the page
headers = {'User-Agent': 'Mozilla/5.0 (Educational Bot)'}  # polite header
response = requests.get(WIKI_URL, headers=headers)

print(f"URL: {WIKI_URL}")
print(f"Status: {response.status_code}")
print(f"HTML size: {len(response.text):,} characters")

URL: https://en.wikipedia.org/wiki/Natural_language_processing
Status: 200
HTML size: 323,426 characters


In [11]:
# Step 2: Parse the HTML
soup = BeautifulSoup(response.text, 'html.parser')

# Step 3: Get the article title
title = soup.find('h1', id='firstHeading').get_text(strip=True)
print(f"Article Title: {title}")

# Step 4: Find the main content area
content_div = soup.find('div', class_='mw-parser-output')

# Step 5: Extract all paragraphs
paragraphs = content_div.find_all('p')
print(f"\nTotal <p> tags found: {len(paragraphs)}")

# Preview first 3 paragraphs
print("\nFirst 3 paragraphs (raw):")
for i, p in enumerate(paragraphs[:3]):
    print(f"\n[Para {i+1}]:\n{p.get_text(strip=True)[:200]}...")

Article Title: Natural language processing

Total <p> tags found: 21

First 3 paragraphs (raw):

[Para 1]:
...

[Para 2]:
Natural language processing(NLP) is the processing ofnatural languageinformation by acomputer. NLP is a subfield ofcomputer scienceand is closely associated withartificial intelligence. NLP is also re...

[Para 3]:
Major processing tasks in an NLP system include:speech recognition,text classification,natural language understanding, andnatural language generation....


In [12]:
import re

# Step 6: Clean and join all paragraph text
raw_paragraphs = []
for p in paragraphs:
    text = p.get_text(strip=True)
    # Skip very short paragraphs (navigation, notes, etc.)
    if len(text) > 50:
        raw_paragraphs.append(text)

print(f"Paragraphs after filtering short ones: {len(raw_paragraphs)}")

# Step 7: Remove Wikipedia citation markers like [1], [2], [citation needed]
def remove_citations(text):
    text = re.sub(r'\[\d+\]', '', text)          # [1], [23]
    text = re.sub(r'\[citation needed\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[note \d+\]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

clean_paragraphs = [remove_citations(p) for p in raw_paragraphs]

# Step 8: Join all into one full document
full_document = '\n\n'.join(clean_paragraphs)

print(f"\n✅ Document extracted!")
print(f"Total characters: {len(full_document):,}")
print(f"Total paragraphs: {len(clean_paragraphs)}")
print(f"\nFirst 500 characters:")
print(full_document[:500])

Paragraphs after filtering short ones: 19

✅ Document extracted!
Total characters: 6,024
Total paragraphs: 19

First 500 characters:
Natural language processing(NLP) is the processing ofnatural languageinformation by acomputer. NLP is a subfield ofcomputer scienceand is closely associated withartificial intelligence. NLP is also related toinformation retrieval,knowledge representation,computational linguistics, andlinguisticsmore broadly.

Major processing tasks in an NLP system include:speech recognition,text classification,natural language understanding, andnatural language generation.

Natural language processing has its r


In [13]:
# BONUS: Also extract section headings
headings = content_div.find_all(['h2', 'h3'])
section_titles = []
for h in headings:
    # Remove the "[edit]" link text Wikipedia adds
    for edit_link in h.find_all('span', class_='mw-editsection'):
        edit_link.decompose()  # remove the tag from soup
    title_text = h.get_text(strip=True)
    if title_text not in ['Contents', 'References', 'External links', 'See also']:
        section_titles.append((h.name, title_text))

print("📑 Article Sections:")
for tag, title in section_titles[:15]:
    indent = '  ' if tag == 'h3' else ''
    print(f"  {indent}[{tag}] {title}")

📑 Article Sections:
  [h2] History
    [h3] Symbolic NLP (1950s – early 1990s)
    [h3] Statistical NLP (1990s–present)
  [h2] Approaches: Symbolic, statistical, neural networks
    [h3] Statistical approach
    [h3] Neural networks
  [h2] Common NLP tasks
    [h3] Text and speech processing
    [h3] Morphological analysis
    [h3] Syntactic analysis
    [h3] Lexical semantics (of individual words in context)
    [h3] Relational semantics (semantics of individual sentences)
    [h3] Discourse (semantics beyond individual sentences)
    [h3] Higher-level NLP applications
  [h2] General tendencies and (possible) future directions


---
# 🧹 PART 3 — NLP Preprocessing Pipeline

Now we clean the scraped Wikipedia text using our full NLP pipeline.

The sequence:
```
1. Lowercase
2. Remove special characters & citations
3. Normalize whitespace
4. Tokenize
5. Remove stop words
6. Lemmatize
```

> 📝 We keep the **original paragraphs** for display, and create **preprocessed versions** for indexing and search.

In [14]:
import re
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

print(f"Stop words loaded: {len(stop_words)} words")
print("Sample stop words:", sorted(list(stop_words))[:15])

Stop words loaded: 198 words
Sample stop words: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't"]


In [15]:
def preprocess_text(text, method='lemmatize'):
    """
    Full NLP preprocessing pipeline.
    method = 'lemmatize' or 'stem'
    Returns cleaned string of tokens.
    """
    # 1. Lowercase
    text = text.lower()

    # 2. Remove citation markers like [1], [23]
    text = re.sub(r'\[\d+\]', '', text)

    # 3. Remove special characters (keep letters, numbers, spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # 4. Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # 5. Tokenize
    tokens = word_tokenize(text)

    # 6. Remove stop words and very short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]

    # 7. Lemmatize or Stem
    if method == 'lemmatize':
        tokens = [lemmatizer.lemmatize(t, pos='v') for t in tokens]
    elif method == 'stem':
        tokens = [stemmer.stem(t) for t in tokens]

    return ' '.join(tokens)


# Test it on a sample paragraph
sample = clean_paragraphs[0]
print("ORIGINAL:")
print(sample[:300])
print()
print("PREPROCESSED (lemmatize):")
print(preprocess_text(sample, method='lemmatize')[:300])
print()
print("PREPROCESSED (stem):")
print(preprocess_text(sample, method='stem')[:300])

ORIGINAL:
Natural language processing(NLP) is the processing ofnatural languageinformation by acomputer. NLP is a subfield ofcomputer scienceand is closely associated withartificial intelligence. NLP is also related toinformation retrieval,knowledge representation,computational linguistics, andlinguisticsmore

PREPROCESSED (lemmatize):
natural language processingnlp process ofnatural languageinformation acomputer nlp subfield ofcomputer scienceand closely associate withartificial intelligence nlp also relate toinformation retrievalknowledge representationcomputational linguistics andlinguisticsmore broadly

PREPROCESSED (stem):
natur languag processingnlp process ofnatur languageinform acomput nlp subfield ofcomput scienceand close associ withartifici intellig nlp also relat toinform retrievalknowledg representationcomput linguist andlinguisticsmor broadli


In [16]:
# Compare lemmatization vs stemming on specific words from the article
test_words = ['processing', 'applications', 'researchers', 'understanding',
              'algorithms', 'computational', 'languages', 'classification']

print(f"{'Word':<20} {'Lemmatized':<20} {'Stemmed'}")
print("-" * 55)
for w in test_words:
    lem_result = lemmatizer.lemmatize(w, pos='v')
    stem_result = stemmer.stem(w)
    print(f"{w:<20} {lem_result:<20} {stem_result}")

Word                 Lemmatized           Stemmed
-------------------------------------------------------
processing           process              process
applications         applications         applic
researchers          researchers          research
understanding        understand           understand
algorithms           algorithms           algorithm
computational        computational        comput
languages            languages            languag
classification       classification       classif


In [17]:
# Preprocess ALL paragraphs
print("Preprocessing all paragraphs...")

preprocessed_paragraphs = []
for i, para in enumerate(clean_paragraphs):
    processed = preprocess_text(para, method='lemmatize')
    preprocessed_paragraphs.append(processed)

print(f"✅ Done! Preprocessed {len(preprocessed_paragraphs)} paragraphs.")

# Quick stats
avg_len = sum(len(p.split()) for p in preprocessed_paragraphs) / len(preprocessed_paragraphs)
print(f"Average tokens per preprocessed paragraph: {avg_len:.0f}")
print(f"\nSample — Original vs Preprocessed:")
print(f"Original  ({len(clean_paragraphs[1].split())} words): {clean_paragraphs[1][:150]}")
print(f"Processed ({len(preprocessed_paragraphs[1].split())} tokens): {preprocessed_paragraphs[1][:150]}")

Preprocessing all paragraphs...
✅ Done! Preprocessed 19 paragraphs.
Average tokens per preprocessed paragraph: 26

Sample — Original vs Preprocessed:
Original  (15 words): Major processing tasks in an NLP system include:speech recognition,text classification,natural language understanding, andnatural language generation.
Processed (13 tokens): major process task nlp system includespeech recognitiontext classificationnatural language understand andnatural language generation


---
# ✂️ PART 4 — Chunking the Document

We have paragraphs — those ARE our chunks. But we'll also build sentence-level chunks for finer retrieval.

We will maintain **two parallel lists**:
- `original_chunks` — the readable, original text (shown to the user)
- `processed_chunks` — the preprocessed text (used for BM25 indexing)

In [18]:
# Strategy: Use paragraphs as chunks (they're already good semantic units)
# But merge very short paragraphs with the next one

def create_chunks(paragraphs, min_words=20):
    """
    Create chunks from paragraphs.
    Merges short paragraphs with the next one.
    """
    chunks = []
    buffer = ""

    for para in paragraphs:
        if len(para.split()) < min_words:
            buffer += " " + para  # merge short para into buffer
        else:
            if buffer:
                chunks.append((buffer.strip() + " " + para).strip())
                buffer = ""
            else:
                chunks.append(para.strip())

    if buffer:  # add any remaining buffer
        chunks.append(buffer.strip())

    return chunks


# Create original chunks (for display)
original_chunks = create_chunks(clean_paragraphs, min_words=20)

# Create processed chunks (for BM25 indexing)
processed_chunks = [preprocess_text(chunk) for chunk in original_chunks]

print(f"Total chunks created: {len(original_chunks)}")
print(f"\nChunk size distribution:")
sizes = [len(c.split()) for c in original_chunks]
print(f"  Min words:  {min(sizes)}")
print(f"  Max words:  {max(sizes)}")
print(f"  Avg words:  {sum(sizes)/len(sizes):.0f}")

print(f"\nSample chunk [0]:")
print(f"  ORIGINAL:  {original_chunks[0][:200]}...")
print(f"  PROCESSED: {processed_chunks[0][:200]}...")

Total chunks created: 17

Chunk size distribution:
  Min words:  20
  Max words:  134
  Avg words:  46

Sample chunk [0]:
  ORIGINAL:  Natural language processing(NLP) is the processing ofnatural languageinformation by acomputer. NLP is a subfield ofcomputer scienceand is closely associated withartificial intelligence. NLP is also re...
  PROCESSED: natural language processingnlp process ofnatural languageinformation acomputer nlp subfield ofcomputer scienceand closely associate withartificial intelligence nlp also relate toinformation retrievalk...


---
# 🔍 PART 5 — BM25 Search Engine

## Quick Recap: Why BM25?

| TF-IDF | BM25 |
|---|---|
| More repetitions = linearly higher score | Diminishing returns after a point |
| Partial length normalization | Full length normalization |
| Good baseline | Better for varied-length docs |

BM25 is the **default algorithm in Elasticsearch**. We'll build the same kind of search engine here.

In [19]:
from rank_bm25 import BM25Okapi

# BM25 needs tokenized input — list of list of words
tokenized_chunks = [chunk.split() for chunk in processed_chunks]

# Build the BM25 index
bm25 = BM25Okapi(tokenized_chunks)

print(f"✅ BM25 index built!")
print(f"   Documents indexed: {len(tokenized_chunks)}")
print(f"   Vocabulary size:   {len(set(w for chunk in tokenized_chunks for w in chunk)):,} unique words")

✅ BM25 index built!
   Documents indexed: 17
   Vocabulary size:   337 unique words


In [20]:
def search(query, top_k=3, show_scores=True):
    """
    Search the BM25 index with a natural language query.
    Returns top_k most relevant chunks.
    """
    # Preprocess the query the same way as the documents
    processed_query = preprocess_text(query)
    query_tokens = processed_query.split()

    if not query_tokens:
        return []

    # Get BM25 scores for all chunks
    scores = bm25.get_scores(query_tokens)

    # Sort by score (highest first)
    import numpy as np
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        if scores[idx] > 0:  # only return relevant results
            results.append({
                'rank': len(results) + 1,
                'score': round(float(scores[idx]), 4),
                'chunk_id': int(idx),
                'text': original_chunks[idx]
            })

    return results


# Test the search
print("🔍 Testing search...\n")
test_query = "machine learning applications"
results = search(test_query, top_k=3)

print(f"Query: '{test_query}'")
print(f"Found {len(results)} results:\n")
for r in results:
    print(f"Rank #{r['rank']} | Score: {r['score']} | Chunk ID: {r['chunk_id']}")
    print(f"Text: {r['text'][:250]}...")
    print("-" * 60)

🔍 Testing search...

Query: 'machine learning applications'
Found 3 results:

Rank #1 | Score: 2.7322 | Chunk ID: 9
Text: Intermediate tasks (e.g., part-of-speech tagging and dependency parsing) are not needed anymore. Neural machine translation, based on then-newly inventedsequence-to-sequencetransformations, made obsolete the intermediate steps, such as word alignment...
------------------------------------------------------------
Rank #2 | Score: 2.5353 | Chunk ID: 5
Text: Machine learningapproaches, which include both statistical and neural networks, on the other hand, have many advantages over the symbolic approach:...
------------------------------------------------------------
Rank #3 | Score: 2.0878 | Chunk ID: 10
Text: The following is a list of some of the most commonly researched tasks in natural language processing. Some of these tasks have direct real-world applications, while others more commonly serve as subtasks that are used to aid in solving larger tasks....
---------

In [21]:
# Try multiple queries to see how the search works
test_queries = [
    "what is natural language processing",
    "history and origin of NLP",
    "deep learning neural networks",
    "text classification sentiment analysis",
    "speech recognition language models"
]

for q in test_queries:
    results = search(q, top_k=1)
    print(f"❓ Query: '{q}'")
    if results:
        print(f"   Score: {results[0]['score']}")
        print(f"   Best match: {results[0]['text'][:180]}...")
    else:
        print("   No relevant results found.")
    print()

❓ Query: 'what is natural language processing'
   Score: 1.5815
   Best match: Up until the 1980s, most natural language processing systems were based on complex sets of hand-written rules. Starting in the late 1980s, however, there was a revolution in natura...

❓ Query: 'history and origin of NLP'
   Score: 0.8578
   Best match: Natural language processing(NLP) is the processing ofnatural languageinformation by acomputer. NLP is a subfield ofcomputer scienceand is closely associated withartificial intellig...

❓ Query: 'deep learning neural networks'
   Score: 5.3071
   Best match: Machine learningapproaches, which include both statistical and neural networks, on the other hand, have many advantages over the symbolic approach:...

❓ Query: 'text classification sentiment analysis'
   No relevant results found.

❓ Query: 'speech recognition language models'
   Score: 2.0516
   Best match: The earliestdecision trees, producing systems of hardif–then rules, were still very similar to the

---
# 🤖 PART 6 — The Wikipedia Chatbot

Now we build the actual chatbot! It will:

1. Accept a question in plain English
2. Preprocess the question
3. Search the BM25 index
4. Return the most relevant passage as the "answer"
5. Handle unknown/irrelevant questions gracefully

> 📝 This is a **retrieval-based chatbot** — it finds and returns relevant text. It doesn't generate new text. That's what RAG + LLM does (Day 5+). But the retrieval part is exactly what we're building now.

In [22]:
import textwrap

def format_answer(text, width=80):
    """Wrap text for neat display."""
    return textwrap.fill(text, width=width)


def chatbot_answer(question, top_k=2, score_threshold=0.5):
    """
    The core chatbot function.
    Takes a question → returns an answer from the Wikipedia article.
    """
    results = search(question, top_k=top_k)

    # Check if we got meaningful results
    if not results or results[0]['score'] < score_threshold:
        return {
            'found': False,
            'answer': "I couldn't find a relevant answer in the article. Try rephrasing your question or asking something more specific to the topic.",
            'score': 0,
            'sources': []
        }

    # Build answer from top results
    best = results[0]
    answer_text = best['text']

    # If second result is also good, add it
    if len(results) > 1 and results[1]['score'] > score_threshold * 0.8:
        answer_text += '\n\n' + results[1]['text']

    return {
        'found': True,
        'answer': answer_text,
        'score': best['score'],
        'sources': [r['chunk_id'] for r in results if r['score'] > score_threshold * 0.8]
    }


def display_response(question, response):
    """Display chatbot response in a readable format."""
    print("=" * 70)
    print(f"👤 YOU: {question}")
    print("-" * 70)
    if response['found']:
        print(f"🤖 BOT (confidence: {response['score']}):")
        print()
        print(format_answer(response['answer'], width=70))
        print()
        print(f"   📎 Source chunk(s): {response['sources']}")
    else:
        print(f"🤖 BOT: {response['answer']}")
    print("=" * 70)
    print()


print("✅ Chatbot functions ready!")

✅ Chatbot functions ready!


In [23]:
# Demo the chatbot with sample questions

demo_questions = [
    "What is natural language processing?",
    "How is machine learning used in NLP?",
    "What are some challenges in NLP?",
    "What is sentiment analysis?",
    "Who invented pizza?"  # off-topic question to test graceful handling
]

for question in demo_questions:
    response = chatbot_answer(question)
    display_response(question, response)

👤 YOU: What is natural language processing?
----------------------------------------------------------------------
🤖 BOT (confidence: 1.5815):

Up until the 1980s, most natural language processing systems were
based on complex sets of hand-written rules. Starting in the late
1980s, however, there was a revolution in natural language processing
with the introduction ofmachine learningalgorithms for language
processing. This shift was influenced by increasing computational
power (seeMoore's law) and a decline in the dominance
ofChomskyanlinguistic theories... (e.g.transformational grammar),
whose theoretical underpinnings discouraged the sort ofcorpus
linguisticsthat underlies the machine-learning approach to language
processing.  Major processing tasks in an NLP system include:speech
recognition,text classification,natural language understanding,
andnatural language generation. Natural language processing has its
roots in the 1950s.Already in 1950,Alan Turingpublished an article
titled 

In [24]:
# 🎮 INTERACTIVE CHATBOT — Run this cell and chat!
# Type your question and press Enter
# Type 'quit' or 'exit' to stop

article_topic = WIKI_URL.split('/wiki/')[-1].replace('_', ' ')

print("=" * 70)
print(f"🤖 Wikipedia Chatbot — Topic: {article_topic}")
print(f"   Indexed {len(original_chunks)} chunks from the article.")
print("   Ask me anything about this topic!")
print("   Type 'quit' to exit.")
print("=" * 70)
print()

while True:
    try:
        question = input("👤 You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n🤖 Bot: Goodbye! Happy learning! 👋")
        break

    if not question:
        continue

    if question.lower() in ['quit', 'exit', 'bye', 'goodbye']:
        print("🤖 Bot: Goodbye! Happy learning! 👋")
        break

    if question.lower() in ['help', '?']:
        print(f"🤖 Bot: I can answer questions about '{article_topic}'. Just ask naturally!")
        print()
        continue

    response = chatbot_answer(question)
    display_response(question, response)

🤖 Wikipedia Chatbot — Topic: Natural language processing
   Indexed 17 chunks from the article.
   Ask me anything about this topic!
   Type 'quit' to exit.

👤 You: whta is nlp
👤 YOU: whta is nlp
----------------------------------------------------------------------
🤖 BOT (confidence: 0.8578):

Natural language processing(NLP) is the processing ofnatural
languageinformation by acomputer. NLP is a subfield ofcomputer
scienceand is closely associated withartificial intelligence. NLP is
also related toinformation retrieval,knowledge
representation,computational linguistics, andlinguisticsmore broadly.
The premise of symbolic NLP is often illustrated usingJohn Searle's
Chinese roomthought experiment: Given a collection of rules (e.g., a
Chinese phrasebook, with questions and matching answers), the computer
emulates natural language understanding (or other NLP tasks) by
applying those rules to the data it confronts.

   📎 Source chunk(s): [0, 2]

👤 You: what is history of nlp
👤 YOU: what is

---
# 🌟 BONUS — Try a Different Wikipedia Article

You can rebuild the entire pipeline for ANY Wikipedia article.
Just run the cell below with a different topic!

In [26]:
def build_chatbot_for_topic(wiki_url):
    """
    Complete pipeline: scrape → preprocess → chunk → index → chatbot.
    """
    print(f"🌐 Fetching: {wiki_url}")

    # 1. Scrape
    headers = {'User-Agent': 'Mozilla/5.0 (Educational Bot)'}
    resp = requests.get(wiki_url, headers=headers)
    soup = BeautifulSoup(resp.text, 'html.parser')

    content_div = soup.find('div', class_='mw-parser-output')
    # Handle case where content_div might not be found, although ZeroDivisionError implies it was found.
    if not content_div:
        print(f"   Could not find main content div ('mw-parser-output') for {wiki_url}.")
        def no_content_found(query, top_k=2, threshold=0.5):
            return "I'm sorry, but no main content could be found for this Wikipedia article."
        return no_content_found

    paragraphs = content_div.find_all('p')

    raw = []
    for p in paragraphs:
        text = remove_citations(p.get_text(strip=True))
        if len(text) > 50:
            raw.append(text)

    print(f"   Scraped {len(raw)} paragraphs.")

    # 2. Chunk
    original_chunks_list = create_chunks(raw, min_words=20) # Renamed to avoid conflict with global original_chunks
    processed_chunks_list = [preprocess_text(c) for c in original_chunks_list]
    print(f"   Created {len(original_chunks_list)} chunks.")

    # --- FIX STARTS HERE ---
    if not original_chunks_list:
        print("   No valid chunks found after processing. Cannot build BM25 index.")
        def no_content_search(query, top_k=2, threshold=0.5):
            return "I'm sorry, but no substantial content was extracted from this article to answer questions."
        return no_content_search
    # --- FIX ENDS HERE ---

    # 3. Build BM25
    tokenized_chunks_list = [chunk.split() for chunk in processed_chunks_list] # Renamed to avoid conflict
    index = BM25Okapi(tokenized_chunks_list)
    print(f"   BM25 index built.")

    # 4. Return search function
    def topic_search(query, top_k=2, threshold=0.5):
        import numpy as np
        pq = preprocess_text(query).split()
        if not pq:
            return "Please provide a valid query." # Handle empty query
        scores = index.get_scores(pq)
        top_idx = np.argsort(scores)[::-1][:top_k]
        results = [{'score': scores[i], 'text': original_chunks_list[i]} # Use original_chunks_list here
                   for i in top_idx if scores[i] > threshold]
        if not results:
            return "I couldn't find an answer in this article."
        return results[0]['text']

    print(f"✅ Chatbot ready! Topic: {wiki_url.split('/wiki/')[-1].replace('_', ' ')}")
    return topic_search


# ✏️ Change the topic below!
# Some ideas:
# https://en.wikipedia.org/wiki/Artificial_intelligence
# https://en.wikipedia.org/wiki/Machine_learning
# https://en.wikipedia.org/wiki/Python_(programming_language)
# https://en.wikipedia.org/wiki/Deep_learning
# https://en.wikipedia.org/wiki/India

new_topic_url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
ai_chatbot = build_chatbot_for_topic(new_topic_url)

# Try it!
answer = ai_chatbot("What is artificial intelligence?")
print("\n🤖 Answer:")
print(format_answer(answer))

🌐 Fetching: https://en.wikipedia.org/wiki/Artificial_intelligence
   Scraped 0 paragraphs.
   Created 0 chunks.
   No valid chunks found after processing. Cannot build BM25 index.

🤖 Answer:
I'm sorry, but no substantial content was extracted from this article to answer
questions.


---
# 📝 Exercises — Your Turn!

Try these on your own to solidify what you've learned.

In [ ]:
# Exercise 1 — BeautifulSoup Practice
# Scrape the Wikipedia article and extract:
#   a) All external links (links that start with 'http')
#   b) All bold text (<b> tags)
#   c) The first image's alt text

soup_ex = BeautifulSoup(response.text, 'html.parser')  # reuse our earlier response

# a) External links
# YOUR CODE HERE
external_links = []
# Hint: soup_ex.find_all('a') and check a.get('href')
print(f"External links found: {len(external_links)}")
print("First 5:", external_links[:5])

print()

# b) All bold text
# YOUR CODE HERE
bold_texts = []
# Hint: find_all('b')
print(f"Bold texts found: {len(bold_texts)}")
print("First 5:", bold_texts[:5])

In [ ]:
# Exercise 2 — Preprocessing Comparison
# Take 3 paragraphs from the article
# Apply preprocessing with BOTH lemmatization and stemming
# Count the unique vocabulary in each approach
# Which one produces a smaller vocabulary? Why?

sample_paras = clean_paragraphs[:3]

# YOUR CODE HERE
lem_tokens = []
stem_tokens = []

# Apply both
for para in sample_paras:
    lem_result = preprocess_text(para, method='lemmatize')
    stem_result = preprocess_text(para, method='stem')
    lem_tokens.extend(lem_result.split())
    stem_tokens.extend(stem_result.split())

print(f"Lemmatization — unique vocab: {len(set(lem_tokens))}")
print(f"Stemming      — unique vocab: {len(set(stem_tokens))}")
print()
print("Why the difference? (your answer here)")

In [ ]:
# Exercise 3 — Build a chatbot for a topic of YOUR choice
# Pick any Wikipedia article and build the full pipeline
# Then ask it at least 5 questions

# ✏️ Fill this in!
MY_TOPIC_URL = "https://en.wikipedia.org/wiki/YOUR_TOPIC_HERE"

# Uncomment and run once you've filled in the URL above:
# my_chatbot = build_chatbot_for_topic(MY_TOPIC_URL)

# Questions to ask:
# my_questions = [
#     "Question 1",
#     "Question 2",
#     "Question 3",
#     "Question 4",
#     "Question 5",
# ]
# for q in my_questions:
#     answer = my_chatbot(q)
#     print(f"Q: {q}")
#     print(f"A: {answer[:300]}...\n")

---
## ✅ What You Built Today

```
Step                    Tool Used            What it Does
──────────────────────────────────────────────────────────────────
Web Scraping            requests             Downloaded the webpage
HTML Parsing            BeautifulSoup        Extracted clean text from HTML
Text Cleaning           re (regex)           Removed citations, special chars
Tokenization            nltk.word_tokenize   Split text into tokens
Stop Word Removal       nltk.stopwords       Removed filler words
Lemmatization           WordNetLemmatizer    Reduced to base word forms
Chunking                Custom function      Split into searchable pieces
Search Indexing         BM25Okapi            Built a keyword search engine
Chatbot                 Python functions     Question → Retrieve → Answer
```

---

## 🚀 What's Coming Next (Day 5)

Right now the chatbot finds answers by **matching keywords**. If you ask about *"cardiac arrest"* but the article says *"heart attack"* — it won't find it.

On Day 5, we replace BM25 with **vector embeddings** — turning every chunk into a dense mathematical vector that captures **meaning**, not just words. Then even *"cardiac arrest"* will match *"heart attack"* because they mean the same thing.

> 💾 **Save this notebook** — you will plug your `clean_paragraphs` and `original_chunks` directly into Day 5's embedding pipeline!